## Decision Tree for Binary Classfication
In a decision tree, we decide if a node will be split or not by looking at the **information gain** that split would give us. 

Where 

$$\text{Information Gain} = H(p_1^\text{node})- \left(w^{\text{left}}H\left(p_1^\text{left}\right) + w^{\text{right}}H\left(p_1^\text{right}\right)\right),$$

and $H$ is the entropy, defined as

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$

In [1]:
import pandas as pd
import numpy as np

In [2]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])


- Ear Shape: Pointy = 1, Floppy = 0
- Face Shape: Round = 1, Not Round = 0
- Whiskers: Present = 1, Absent = 0

Therefore, we have two sets:

- `X_train`: for each example, contains 3 features:
            - Ear Shape (1 if pointy, 0 otherwise)
            - Face Shape (1 if round, 0 otherwise)
            - Whiskers (1 if present, 0 otherwise)
            
- `y_train`: whether the animal is a cat
            - 1 if the animal is a cat
            - 0 otherwise


  ---

In [3]:
#For instance, the first example
X_train[0]

array([1, 1, 1])

This means that the first example has a pointy ear shape, round face shape and it has whiskers.

In [4]:
def compute_entropy(y):

    p = 0
    
    if len(y) == 0:
        return 0
    p = sum(y)/len(y)
    if p == 0 or p == 1:
        return 0
    else:
        return -p*np.log2(p) - (1-p)*np.log2(1-p)
     

print(compute_entropy(y_train))

1.0


In [5]:
def split_dataset(X, node_indices, feature_idx):
    """Given a dataset and a feature_idx, return two lists for the two split nodes, the left node has the animals that have 
    that feature = 1 and the right node those that have the feature = 0 
    feature_idx = 0 => ear shape
    feature_idx = 1 => face shape
    feature_idx = 2 => whiskers
    """

    left_indices = []
    right_indices = []
    

    for i in node_indices:
        if X[i][feature_idx] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)


    return left_indices,right_indices

In [6]:
split_dataset(X_train,[0,1,2,3,4,5,6,7,8,9],0)

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])

In [7]:
def compute_information_gain(X,y,node_indices,feature):
    """
    This function takes the splitted dataset, the indices we chose to split and returns the weighted entropy.
    """

    left_indices,right_indices = split_dataset(X, node_indices, feature)

    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    

    information_gain=0


   
    node_entropy = compute_entropy(y_node)
    left_entropy = compute_entropy(y_left)
    right_entropy = compute_entropy(y_right)
    w_left = len(X_left) / len(X_node)
    w_right = len(X_right) / len(X_node)

   

    weighted_entropy = w_left * left_entropy + w_right * right_entropy


    return node_entropy - weighted_entropy


    

In [8]:
compute_information_gain(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], 0)

np.float64(0.2780719051126377)

In [9]:
def get_best_split(X, y, node_indices):   
    num_features = X.shape[1]
    
    best_feature = -1

    max_info_gain = 0
    for feature in range(num_features):
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_feature = feature
        
   
    return best_feature

In [10]:
def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth, tree):

    # Stopping condition
    if current_depth == max_depth:
        print("    " * current_depth + f"{branch_name} Leaf -> {node_indices}")
        return

    # Find the best feature
    best_feature = get_best_split(X, y, node_indices)

    # Print current node
    print("    " * current_depth +
          f"{branch_name} Node (Depth {current_depth}) -> Split on Feature {best_feature}")

    # Split the dataset
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)

    # Store the split
    tree.append((left_indices, right_indices, best_feature))

    # Recursive calls
    build_tree_recursive(
        X, y, left_indices,
        "Left",
        max_depth,
        current_depth + 1,
        tree
    )

    build_tree_recursive(
        X, y, right_indices,
        "Right",
        max_depth,
        current_depth + 1,
        tree
    )

    return tree

In [11]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)


Root Node (Depth 0) -> Split on Feature 0
    Left Node (Depth 1) -> Split on Feature 1
        Left Leaf -> [0, 4, 5, 7]
        Right Leaf -> [3]
    Right Node (Depth 1) -> Split on Feature 2
        Left Leaf -> [1]
        Right Leaf -> [2, 6, 8, 9]


[([0, 3, 4, 5, 7], [1, 2, 6, 8, 9], 0),
 ([0, 4, 5, 7], [3], 1),
 ([1], [2, 6, 8, 9], 2)]